<a href="https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis + Time Window

Unit of Analysis

One row represents one pseudonymized content item (page/article) from one client in the starter dataset. Each row contains summary metrics describing that content item's performance.

Time Window

All performance metrics are calculated over the trailing 90-day window before the dataset snapshot. The dataset does not contain daily records—each row is a 90-day summary for a single content item.

In [8]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            print(os.path.join(root, file))

./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv


In [12]:
import pandas as pd

df = pd.read_csv("./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())


Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Category     | Fields                                                                                                                                                                             | Reason                                                                                                                                                                                                                        |
| ------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Features** | `clicks`, `impressions`, `ctr`, `engagement_rate`, `scroll_rate`, `word_count`, `keyword_count`, `backlinks`, `internal_links`, `page_speed`, and other historical content metrics | These variables are available before prediction and can be safely used to train the model.                                                                                                                                    |
| **Label**    | `is_declining_label`                                                                                                                                                               | This is the target variable that the model predicts. It must never be used as an input feature.                                                                                                                               |
| **Context**  | `content_id`, `client_id`, `content_type`                                                                                                                                          | These fields are used for identification, grouping, and train/test splitting. They help organize the data but should not be learned by the model.                                                                             |
| **Excluded** | `trend_direction`, `trend_pct`, `content_id`, `client_id`                                                                                                                          | `trend_direction` and `trend_pct` are used to derive `is_declining_label`, so using them would cause **target leakage**. `content_id` and `client_id` are identifiers and do not represent meaningful predictive information. |


#Note
Features: Historical performance metrics known before the prediction.
Label: The outcome the model is trying to predict.
Context: Used for joins, grouping, or splitting the dataset (e.g., grouped train/test split by client_id).
Excluded: Removed because they either leak future information (trend_direction, trend_pct) or are identifiers (content_id, client_id) that should not be model inputs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verify the Grain (One row = one content item)

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Total rows
print("Total rows:", len(df))

# Unique content IDs
print("Unique content IDs:", df["content_id"].nunique())

# Duplicate content IDs
print("Duplicate content IDs:", df["content_id"].duplicated().sum())


Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


Verify Counts

In [15]:
print("Dataset shape:", df.shape)

print("\nNumber of clients:")
print(df["client_id"].nunique())

print("\nContent types:")
print(df["content_type"].value_counts())

Dataset shape: (30000, 44)

Number of clients:
32

Content types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


Verify Missing Values

In [16]:
missing = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
      .mul(100)
      .round(2)
)

print(missing)

provider_used             71.46
word_count                25.66
char_count                25.66
word_count_tier           25.66
char_count_tier           25.66
model_used                19.11
trend_pct                 11.29
competition_level          8.70
search_volume              8.23
cpc                        8.23
competition                8.23
main_intent                7.91
scroll_rate                0.42
content_type               0.00
client_id                  0.00
content_id                 0.00
impressions_90d            0.00
clicks_90d                 0.00
pageviews_90d              0.00
sessions_90d               0.00
days_with_impressions      0.00
days_with_sessions         0.00
impressions_last_30d       0.00
clicks_last_30d            0.00
users_90d                  0.00
engaged_sessions_90d       0.00
ai_sessions_90d            0.00
scroll_events_90d          0.00
sessions_prev_30d          0.00
clicks_prev_30d            0.00
impressions_prev_30d       0.00
sessions

Verify the Time Window

In [18]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



* The dataset contains only **90-day summary metrics**, not daily data.
* Client history is **uneven**, so comparisons across clients may be biased.
* **`trend_pct`** and **`trend_direction`** cannot be used as features because they cause target leakage.
* Missing values are **not random** and depend on `content_type`.
* In the warehouse dataset, early rows may contain only **GSC** data, and overlapping time windows can cause data leakage if not handled carefully.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.